<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/scikit-learn/Employee_Attrition_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ushbu sahifada mini loyiha : Employee Attrition Prediction ya'ni Kompaniya xodimning ishdan ketishini (attrition) bashorat qilamiz . Dataset AI tomonidan yaratildi (real hayotdagi ma'lumotlarga o'xshash qilib) . Target ustun : 0-ishda qoladi , 1-ishdan ketadi.**

In [ ]:
import numpy as np
import pandas as pd
import sklearn

In [ ]:
df = pd.DataFrame({
    "age": [23,45,31,28,39,50,36,29,41,33,
            26,48,37,30,42,27,35,46,32,40],

    "salary": [3000,9000,np.nan,4200,7000,12000,6500,3800,8500,np.nan,
               3400,11000,6800,4500,9200,3600,6000,10500,4800,8800],

    "experience": [1,20,6,3,12,25,10,4,15,8,
                   2,22,11,5,17,3,9,21,7,16],

    "department": [
        "IT","HR","Sales","IT","Finance","Management","IT","Sales","Finance","HR",
        "IT","Management","Finance","Sales","Management","HR","IT","Management","Sales","Finance"
    ],

    "education": [
        "Bachelor","Master","Bachelor","Bachelor","Master","PhD","Bachelor","Bachelor","Master","Bachelor",
        "Bachelor","PhD","Master","Bachelor","Master","Bachelor","Bachelor","PhD","Bachelor","Master"
    ],

    "attrition": [
        1,0,1,1,0,0,0,1,0,1,
        1,0,0,1,0,1,0,0,1,0
    ]
})
df.head()

,age,salary,experience,department,education,attrition
0,23,3000.0,1,IT,Bachelor,1
1,45,9000.0,20,HR,Master,0
2,31,NaN,6,Sales,Bachelor,1
3,28,4200.0,3,IT,Bachelor,1
4,39,7000.0,12,Finance,Master,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         20 non-null     int64  
 1   salary      18 non-null     float64
 2   experience  20 non-null     int64  
 3   department  20 non-null     object 
 4   education   20 non-null     object 
 5   attrition   20 non-null     int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 1.1+ KB


In [ ]:
# featuresni ajratish
X = df.drop('attrition', axis=1)
y = df['attrition']

# sonli va matnli ustunlarni ajratish
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=np.object_).columns.tolist()

cat_cols

['department', 'education']

In [ ]:
num_cols

['age', 'salary', 'experience']

In [ ]:
print(X.columns.tolist())

['age', 'salary', 'experience', 'department', 'education']


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# sonli pipeline yaratish
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# matnli pipeline yaratish
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer yaratish
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
# umumiy pipeline yaratish
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

scores = cross_val_score(full_pipe, X, y, cv=5, scoring='accuracy')
print(scores)
print("scores mean:", scores.mean())
print("scores std:", scores.std())

[1.   1.   1.   1.   0.75]
scores mean: 0.95
scores std: 0.09999999999999999


- LogisticRegression modelimiz 5 ta fold bo'yicha o'rtacha 95% accuracy ko'rsatmoqda
- Std=0.10 bo'lgani esa , har bir fold natijalari biroz o'zgarib turishini bildiradi , rostdan ham ohirgi folddagi natija 0.75 bo'ldi

In [16]:
scores = cross_val_score(full_pipe, X, y, cv=5, scoring='precision')
print(scores)
print("scores mean:", scores.mean())
print("scores std:", scores.std())

[1.  1.  1.  1.  0.5]
scores mean: 0.9
scores std: 0.2


- model ishdan ketadi deganlarning 90% haqiqatdan ham ishdan ketgan , ba'zida model ketadi lekin aslida ketmaganlar ham bor

In [17]:
scores = cross_val_score(full_pipe, X, y, cv=5, scoring='recall')
print(scores)
print("scores mean:", scores.mean())
print("scores std:", scores.std())

[1. 1. 1. 1. 1.]
scores mean: 1.0
scores std: 0.0


- model - barcha haqiqiy attrition=1 bo'lgan xodimlarni topa olgan , ya'ni ishdan ketadigan xodimlarni
- std=0 bo'lgani esa - hamma foldlarning natijalari bir xil bo'lgan

In [18]:
scores = cross_val_score(full_pipe, X, y, cv=5, scoring='f1')
print(scores)
print("scores mean:", scores.mean())
print("scores std:", scores.std())

[1.         1.         1.         1.         0.66666667]
scores mean: 0.9333333333333333
scores std: 0.13333333333333336


- bu f1-score esa precision va recallning o'rtachasidur